# Show the `lsst.analysis.tools` plots produced by DRP stage1 (single-visit)

- author : Sylvie Dagoret-Campagne
- creation date : 2026-11-06 : stage 1
- purpose : display the plots made by the `lsst.analysis.tools` tasks of `DRP.yaml#stage1-single-visit` (dataset types ending with `Plot`), to understand what the steps (1c, 1d) do
- helpers : `showplots_utils.py` (same folder), shared with the stage 2 notebook
- next : `nb2026-11-06_ShowPlots_DRPstage2-recalibrate.ipynb`

- the quantum graph for DRP.yaml#stage1-single-visit: https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_40/drp_pipe/LSSTCam/DRP/pipeline_drp_pipe_LSSTCam_DRP_stage1-single-visit.pdf
- all quantum graphs : https://tigress-web.princeton.edu/~lkelvin/pipelines/2026/w_2026_43/drp_pipe/LSSTCam/DRP/

## Import

In [ ]:
import os
import re
import logging
import warnings
import importlib
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

from lsst.daf.butler import Butler

import showplots_utils
importlib.reload(showplots_utils)   # pick up any edit of the helper module without restarting the kernel
from showplots_utils import PlotBrowser, show_diagram

In [ ]:
# Turn the Butler.collections FutureWarning into an error (same convention as the CheckCollections notebooks)
warnings.filterwarnings(
    'error',
    category=FutureWarning,
    message='.*Butler.collections.*',
)

logging.basicConfig(
    level=logging.WARNING,
    format='%(name)s %(levelname)s: %(message)s',
    force=True,
)

butler = Butler('main')

## Configuration

In [ ]:
NB_TAG = 'ShowPlots_DRPstage1'

username = os.getenv('USER')
RUN_NAME = 'runcosmos01_20261103_tract9813patchsel_RunCosmos9patches'
collection_path = f'u/{username}/LSSTCam/{RUN_NAME}/stage1'

# Plot files retrieved from the Butler (a file already present is not downloaded again)
FIGDIR = Path(f'figs_{NB_TAG}')

# Quantum diagrams of the pipeline steps
DIAGRAM_DIR = Path('../../../../notebooks/fig_diagrams')

print('collection :', collection_path)

## Quantum diagrams of the stage1 steps

In [ ]:
DIAGRAM_STEPS = {
    'step1a-single-visit-detectors': 'DRP step1a/detectors',
    'step1b-single-visit-visits':    'DRP step1b/visits',
    'step1c-single-visit-tracts':    'DRP step1c/tracts',
    'step1d-single-visit-global':    'DRP step1d/global',
}

# Steps to display (the plots come from 1c and 1d) ; add 1a / 1b if needed
SHOW_STEPS = ['step1c-single-visit-tracts', 'step1d-single-visit-global']

for step in SHOW_STEPS:
    show_diagram(DIAGRAM_DIR / f'pipeline_drp_pipe_LSSTCam_DRP_{step}.png', title=DIAGRAM_STEPS[step])

## Plots in the stage1 collection

Only the dataset types matching `*Plot` are queried (much faster than listing the ~44 million datasets of the collection).

In [ ]:
pb = PlotBrowser(butler, collection_path, raw_dir=FIGDIR)

print('collection            :', collection_path)
print('number of plots       :', len(pb.df))
print('common name prefix    :', pb.prefix)

with pd.option_context('display.max_colwidth', None, 'display.width', 250):
    display(pb.summary())

### Reading the dimensions

- most plots have dimensions `band, instrument, skymap, tract` : one plot **per band and per tract** (tract-level step, presumably step 1c). There is no `visit` dimension : the star association gathers all the visits of the tract in a band.
- the `whole_sky` plots have dimensions `band, skymap` : one plot **per band over the whole skymap** (global step, presumably step 1d).

Completeness check : number of plots per dataset type and per band (all the cells should be equal to the number of tracts).

In [ ]:
display(pb.completeness('band'))

## Families of plots

Each family is selected by a regular expression on the dataset type name (prefix removed). `pb.show(pattern)` displays one **montage per dataset type, one panel per band**. Use `mode='single'` to see each plot alone at full resolution, and `band='i'` to restrict to one band.

The descriptions are a reading guide only : the exact definition of each metric is in the `lsst.analysis.tools` code, and the title of each plot gives the details.

In [ ]:
FAMILY_PATTERNS = {
    'astrom_repeat':       r'stellarAstrometricRepeatability[0-9]_HistPlot',
    'astrom_selfrepeat':   r'stellarAstrometricSelfRepeatability(?:RA|Dec)_HistPlot',
    'astrom_resid_fp':     r'stellarAstrometricResiduals(?:RA|Dec)_FocalPlanePlot',
    'astrom_resid_std_fp': r'stellarAstrometricResidualStdDev(?:RA|Dec)_FocalPlanePlot',
    'photo_repeat':        r'stellarPhotometricRepeatability(?:Calib)?_HistPlot',
    'photo_resid_fp':      r'stellarPhotometricResiduals(?:Calib)?_FocalPlanePlot',
    'whole_sky':           r'^whole_sky_',
}

# Dataset types not covered by a family (new plots appearing in a newer release, ...)
unmatched = [s for s in pb.summary().index
             if not any(re.search(p, s) for p in FAMILY_PATTERNS.values())]
print('Dataset types without family :', unmatched)

### Astrometric repeatability (histograms)

Distribution of the astrometric repeatability of stars (scatter of repeated position measurements). The digits 1, 2, 3 label the variants of the metric.

In [ ]:
pb.show(FAMILY_PATTERNS['astrom_repeat'])

### Astrometric self-repeatability, RA and Dec (histograms)

Same idea, separately for the RA and Dec coordinates.

In [ ]:
pb.show(FAMILY_PATTERNS['astrom_selfrepeat'])

### Astrometric residuals in the focal plane, RA and Dec

Astrometric residuals binned in focal-plane coordinates : position-dependent systematics (WCS, distortion).

In [ ]:
pb.show(FAMILY_PATTERNS['astrom_resid_fp'])

### Astrometric residual standard deviation in the focal plane, RA and Dec

Dispersion of the astrometric residuals across the focal plane.

In [ ]:
pb.show(FAMILY_PATTERNS['astrom_resid_std_fp'])

### Photometric repeatability (histograms)

Distribution of the photometric repeatability of stars. The `Calib` variant is the calibrated-flux version.

In [ ]:
pb.show(FAMILY_PATTERNS['photo_repeat'])

### Photometric residuals in the focal plane

Photometric residuals binned in focal-plane coordinates : detector- or position-dependent photometric systematics.

In [ ]:
pb.show(FAMILY_PATTERNS['photo_resid_fp'])

### Whole-sky maps (step 1d)

Sky maps of the astrometric repeatability metrics (AD1, AM2). With a COSMOS-only run only a small area of the sky is filled.

In [ ]:
pb.show(FAMILY_PATTERNS['whole_sky'])

## Zoom : one band, full resolution

Every plot of one band, one after the other (change `BAND`, or add a `pattern` to restrict to a family).

In [ ]:
BAND = 'i'

pb.show(mode='single', band=BAND)
# only one family :
# pb.show(FAMILY_PATTERNS['photo_repeat'], mode='single', band=BAND)

## Beyond the plots : the metrics behind them

The plots are made from the same star association as the metric datasets of the collection. List them, and read one.

In [ ]:
metric_types = pb.sibling_types('metric')
print(f'{len(metric_types)} metric dataset type(s) in the collection :')
for name in metric_types:
    print('  ', name)

In [ ]:
# Pick one of the types listed above
METRIC_TYPE = metric_types[0] if metric_types else None

if METRIC_TYPE is not None:
    metrics = pb.load_first(METRIC_TYPE)
    if hasattr(metrics, 'to_pandas'):
        display(metrics.to_pandas().head())
    else:
        print(str(metrics)[:2000])